# InferDoc closed loop experiment

Flagship hackathon flow: **INFER → MEASURE → ANALYZE → DIAGNOSE → RECOMMEND → VALIDATE → RERUN → VERIFY**. Nebius Token Factory serves NVIDIA Nemotron-3 Nano. Nemotron proposes; Python measures, admits, and verifies.

## Configuration and controlled baseline

This notebook calls the reusable SDK workflow once. It spends Nebius credits only when `INFERDOC_RUN_LIVE_TESTS=1` and `NEBIUS_API_KEY` are set. Without the gate, run notebook 04 for a fully offline replay.

In [ ]:
import json
import os
from inferdoc.benchmark.workload import WorkloadSpec
from inferdoc.config import InferDocSettings
from inferdoc.nebius.client import NebiusClient
from inferdoc.storage import ArtifactStore
from inferdoc.workflows import run_closed_loop

settings = InferDocSettings.from_env()
live_enabled = os.getenv("INFERDOC_RUN_LIVE_TESTS") == "1" and bool(settings.api_key)
workload = WorkloadSpec(
    prompts=["Define TTFT in one sentence.", "Define throughput in one sentence.",
             "Explain p95 latency in one sentence.", "Explain concurrency in one sentence."],
    concurrency=1, max_tokens=32, temperature=0.0, stream=False, enable_thinking=False,
)
store = ArtifactStore(settings.artifact_dir)
print({"model": settings.default_model, "request_count": len(workload.prompts),
       "live_enabled": live_enabled})

## Run the SDK workflow

The workflow performs baseline measurement, Nemotron diagnosis, deterministic admission, a candidate rerun when admitted, and deterministic verification. No notebook cell reimplements those steps.

In [ ]:
result = None
if live_enabled:
    client = NebiusClient(settings=settings)
    try:
        result = await run_closed_loop(
            client=client, workload=workload, model=settings.default_model, artifact_store=store
        )
    finally:
        await client.aclose()
else:
    print("Live run skipped. Use 04_offline_replay_and_audit.ipynb for no-credit replay.")

## Baseline EvidenceBundle and Nemotron diagnosis

In [ ]:
if result is not None:
    print({"baseline_run_id": result.baseline.run_id,
           "aggregate": result.baseline.aggregate,
           "capabilities": result.baseline.capabilities.model_dump()})
    print({"diagnosis": result.diagnosis.recommendation_summary,
           "experiment": result.diagnosis.experiment.model_dump(mode="json") if result.diagnosis.experiment else None})
else:
    print("Awaiting an explicitly enabled live run.")

## ExperimentSpec, deterministic admission, and candidate rerun

In [ ]:
if result is not None:
    print({"admission": result.admission.model_dump(mode="json") if result.admission else None,
           "candidate_run_id": result.candidate.run_id if result.candidate else None,
           "candidate_aggregate": result.candidate.aggregate if result.candidate else None})
else:
    print("Awaiting an explicitly enabled live run.")

## Deterministic verification and artifact hashes

PASS, FAIL, and INCONCLUSIVE are all valid scientific outcomes. No speedup is assumed. When no executable experiment is proposed, the typed result has no candidate or verification.

In [ ]:
if result is not None:
    print({"verification": result.verification.model_dump(mode="json") if result.verification else None,
           "baseline_artifacts": str(store.root / result.baseline.run_id),
           "baseline_manifest": store.verify_manifest(result.baseline.run_id).model_dump(),
           "baseline_hashes": json.loads((store.root / result.baseline.run_id / "manifest.json").read_text(encoding="utf-8"))["artifacts"]})
    if result.candidate is not None:
        print({"candidate_artifacts": str(store.root / result.candidate.run_id),
               "candidate_manifest": store.verify_manifest(result.candidate.run_id).model_dump(),
               "candidate_hashes": json.loads((store.root / result.candidate.run_id / "manifest.json").read_text(encoding="utf-8"))["artifacts"]})
else:
    print("Awaiting an explicitly enabled live run.")